# 03 · M3 — Keypoints de llantas y número de ejes (YOLO-pose)
**IAM-811** · Laboratorio Final de Hito

Basado en `keypoints_llantas_yolo_detectron2_v3.ipynb` del curso. Cada llanta tiene **5 keypoints en este orden**:

| índice | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| keypoint | `centro` | `contacto` (con el piso) | `superior` | `izquierda` | `derecha` |

`kpt_shape: [5, 3]` y `flip_idx: [0, 1, 2, 4, 3]` (al voltear se intercambian izquierda y derecha).
Con ellos el servidor: descarta las llantas del lado lejano, agrupa centros a menos de 0.6 diámetros
(ruedas dobles = 1 eje) y cuenta los grupos = **número de ejes**.

**Contrato con `/inferencia`** (sección 6): archivo **`llantas_pose.pt`**, `task = pose`, `kpt_shape = [5, 3]`,
una clase (`llanta`). El orden de los keypoints no se puede verificar desde el `.pt`: debe ser el de la tabla.

**GPU T4** (Colab) o el mismo notebook en un Pod de RunPod.

In [ ]:
!nvidia-smi -L || echo "Sin GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU T4"
!pip -q install -U ultralytics==8.4.218 roboflow==1.7.3

## 1. Dataset
Roboflow: proyecto *Keypoint Detection*, clase `llanta`, esqueleto con los 5 puntos **en el orden de la tabla**
(Roboflow exporta los keypoints en el orden en que los definiste). Exporta en formato **YOLOv11**.
Alternativa: el zip `dataset_llantas.zip` del notebook 01 (ya viene en el orden correcto).

In [ ]:
import os, glob, yaml, shutil, numpy as np, cv2
FUENTE = 'roboflow'          # 'roboflow' o 'zip'
WORKSPACE, PROYECTO, VERSION = 'tu-workspace', 'peaje-llantas', 1
KPT_NOMBRES = ['centro', 'contacto', 'superior', 'izquierda', 'derecha']
KPT_SHAPE = [5, 3]
FLIP_IDX = [0, 1, 2, 4, 3]
NOMBRE_FINAL = 'llantas_pose.pt'      # nombre EXACTO que busca inferencia/app/config.py

def preparar_yaml(ruta_yaml):
    """Deja el data.yaml con rutas absolutas validas en Colab (Roboflow exporta '../train/images', 'valid/', etc.)."""
    d = yaml.safe_load(open(ruta_yaml))
    base = os.path.dirname(os.path.abspath(ruta_yaml))
    d['path'] = base
    for clave, candidatas in (('train', ['train/images', 'images/train']),
                              ('val', ['valid/images', 'val/images', 'images/val']),
                              ('test', ['test/images', 'images/test'])):
        encontrada = next((c for c in candidatas if os.path.isdir(os.path.join(base, c))), None)
        if encontrada:
            d[clave] = encontrada
        elif clave == 'test':
            d.pop('test', None)
        else:
            raise FileNotFoundError(f'No hay carpeta {clave} en {base}. En Roboflow genera la version con split train/valid.')
    yaml.safe_dump(d, open(ruta_yaml, 'w'), allow_unicode=True, sort_keys=False)
    return d

if FUENTE == 'roboflow':
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
    ds = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION).download('yolov11', location='/content/ds_llantas')
    DATA_YAML = f'{ds.location}/data.yaml'
else:
    from google.colab import files
    for n in files.upload():
        !unzip -qo "$n" -d /content
    DATA_YAML = '/content/dataset_llantas/data.yaml'

datos = preparar_yaml(DATA_YAML)
# Roboflow escribe kpt_shape segun el esqueleto: si no es [5, 3] el proyecto esta mal definido
kpt_export = datos.get('kpt_shape')
assert kpt_export in (None, KPT_SHAPE), f'El dataset trae kpt_shape={kpt_export}; el proyecto debe tener 5 keypoints.'
# Verifica una etiqueta: 1 clase + 4 de caja + 5*3 de keypoints = 20 valores por linea
ejemplo = next((l for t in glob.glob(os.path.join(datos['path'], datos['train'].replace('images', 'labels'), '*.txt'))
                for l in open(t).read().split('\n') if l.strip()), '')
assert len(ejemplo.split()) == 20, f'Cada línea de etiqueta debe tener 20 valores (tiene {len(ejemplo.split())}): {ejemplo[:80]}'
datos['kpt_shape'] = KPT_SHAPE
datos['flip_idx'] = FLIP_IDX
yaml.safe_dump(datos, open(DATA_YAML, 'w'), sort_keys=False)
print(datos)

## 2. Entrenamiento YOLO-pose
`yolo11s-pose.pt` trae un cabezal de 17 keypoints humanos; Ultralytics lo adapta a `kpt_shape [5, 3]`.
`imgsz=640` = `IMGSZ_PROPIO` del servidor.

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11s-pose.pt')
modelo.train(data=DATA_YAML, epochs=80, imgsz=640, batch=16, patience=20, device=0, fliplr=0.5,
             project='/content/runs/pose', name='llantas', exist_ok=True)
BEST = '/content/runs/pose/llantas/weights/best.pt'
m = YOLO(BEST).val(data=DATA_YAML)
print(f'mAP50 keypoints: {m.pose.map50:.3f} | mAP50-95 keypoints: {m.pose.map:.3f} | mAP50 cajas: {m.box.map50:.3f}')

## 3. Conteo de ejes (misma lógica que `inferencia/app/ejes.py`)

In [ ]:
UMBRAL_CONTENCION, UMBRAL_CERCANA, SEPARACION_EJE = 0.7, 0.65, 0.6
modelo_llantas = YOLO(BEST)

def altura_visible(l):
    return max(1.0, l['kps'][1, 1] - l['kps'][2, 1])

def contenida(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    return max(0, x2 - x1) * max(0, y2 - y1) / max(1e-6, (a[2] - a[0]) * (a[3] - a[1]))

def contar_ejes(llantas):
    orden = sorted(llantas, key=lambda l: -(l['box'][2] - l['box'][0]) * (l['box'][3] - l['box'][1]))
    ls = []
    for l in orden:
        if all(contenida(l['box'], r['box']) < UMBRAL_CONTENCION for r in ls):
            ls.append(l)
    if not ls:
        return None
    hmax = max(altura_visible(l) for l in ls)
    cercanas = [l for l in ls if altura_visible(l) >= UMBRAL_CERCANA * hmax]
    xs = sorted(l['kps'][0, 0] for l in cercanas)
    diam = float(np.median([altura_visible(l) for l in cercanas]))
    return max(2, 1 + sum(1 for p, q in zip(xs, xs[1:]) if q - p > SEPARACION_EJE * diam))

def llantas_pose(img, conf=0.3):
    r = modelo_llantas.predict(img, conf=conf, imgsz=640, verbose=False)[0]
    out = []
    if r.keypoints is None:
        return out, r
    for b, p, c in zip(r.boxes.xyxy.cpu().numpy(), r.keypoints.xy.cpu().numpy(), r.keypoints.conf.cpu().numpy()):
        out.append({'box': b.tolist(), 'kps': np.concatenate([p, c[:, None]], 1)})
    return out, r

## 4. (Opcional) Evaluación de ejes sobre imágenes con conteo conocido
Pon `EVALUAR_EJES = True` y sube un CSV `ejes_reales.csv` (`archivo,ejes`, vistas laterales) junto con las imágenes o un zip.

In [ ]:
import pandas as pd
EVALUAR_EJES = False
if EVALUAR_EJES:
    from google.colab import files
    for n in files.upload():
        if n.endswith('.zip'):
            !unzip -qo "$n" -d /content/eval_ejes
    gt = pd.read_csv('ejes_reales.csv')
    filas = []
    for _, f in gt.iterrows():
        ruta = f['archivo'] if os.path.exists(f['archivo']) else os.path.join('/content/eval_ejes', f['archivo'])
        img = cv2.imread(ruta)
        if img is None:
            continue
        filas.append({'archivo': f['archivo'], 'real': f['ejes'], 'predicho': contar_ejes(llantas_pose(img)[0])})
    res = pd.DataFrame(filas)
    res['acierto'] = res['real'] == res['predicho']
    print('Exactitud de ejes:', round(res['acierto'].mean() * 100, 1), '%',
          '| MAE:', round((res['real'] - res['predicho'].fillna(2)).abs().mean(), 2))
    display(res)

## 5. Visualización

In [ ]:
import matplotlib.pyplot as plt
!wget -q -O prueba.jpg https://ultralytics.com/images/bus.jpg
ls, r = llantas_pose(cv2.imread('prueba.jpg'))
print('Llantas:', len(ls), '| ejes estimados:', contar_ejes(ls))
plt.figure(figsize=(10, 7)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis('off'); plt.show()

## 6. Verificar contrato y exportar `llantas_pose.pt`
(Detectron2 Keypoint R-CNN está en la sección 8.4 del notebook del curso; el servidor usa YOLO-pose por velocidad.)

In [ ]:
final = YOLO(BEST)
kpt = list(getattr(final.model, 'kpt_shape', None) or final.model.yaml.get('kpt_shape'))
assert final.task == 'pose', f'task={final.task}: debe ser pose'
assert kpt == KPT_SHAPE, f'kpt_shape={kpt}: debe ser {KPT_SHAPE}'
print('OK contrato M3 · task=pose · kpt_shape =', kpt, '· clases =', final.names)

shutil.copy(BEST, f'/content/{NOMBRE_FINAL}')
GUARDAR_DRIVE = True
if GUARDAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/peaje/pesos', exist_ok=True)
    shutil.copy(f'/content/{NOMBRE_FINAL}', f'/content/drive/MyDrive/peaje/pesos/{NOMBRE_FINAL}')
    print('Guardado en Drive: MyDrive/peaje/pesos/' + NOMBRE_FINAL)
from google.colab import files
files.download(f'/content/{NOMBRE_FINAL}')